In [2]:
import pygame
import math
import numpy as np
import cvxpy as cp

# --- CVXPY (MPC) Setup ---
N = 20              # Prediction Horizon: How many steps to plan ahead
DT = 0.1            # Time step for the plan
MAX_SPEED = 50      # Max speed (pixels per second)
MIN_DIST = 40       # Min distance from obstacle (radius + obstacle size)

# Create the CVXPY optimization variables
x_var = cp.Variable((N + 1, 2))
u_var = cp.Variable((N, 2))
prob = None

# Define CVXPY Parameters
start_pos_param = cp.Parameter(2, name="start_pos")
target_pos_param = cp.Parameter(2, name="target_pos")
a_param = cp.Parameter(name="a")
b_param = cp.Parameter(name="b")
c_param = cp.Parameter(name="c")


def solve_mpc_trajectory(start_pos, target_pos, obstacle_center):
    """
    Solves the MPC optimization problem for one frame.
    Finds the best path from start_pos to target_pos while avoiding the obstacle.
    """
    global prob
    
    # --- The "Big Trick": Linearizing the Obstacle Constraint ---
    obs_to_obj_vec = start_pos - obstacle_center
    norm = np.linalg.norm(obs_to_obj_vec)
    if norm < 1e-6:
        # Failsafe if object is right on the obstacle center
        obs_to_obj_vec = np.array([1.0, 0.0])
        norm = 1.0
        
    a = obs_to_obj_vec[0] / norm
    b = obs_to_obj_vec[1] / norm
    
    # Calculate 'c' (the constant for the line equation)
    c = a * (obstacle_center[0] + a * MIN_DIST) + b * (obstacle_center[1] + b * MIN_DIST)
    
    # --- Update Parameter Values ---
    start_pos_param.value = start_pos
    target_pos_param.value = target_pos
    a_param.value = a
    b_param.value = b
    c_param.value = c

    # --- Define the Problem (ONLY ONCE) ---
    if prob is None:
        # --- Define the Cost Function (using Parameters) ---
        cost = cp.sum_squares(x_var[N] - target_pos_param)
        cost += 0.5 * cp.sum_squares(u_var)

        # --- Define the Constraints (using Parameters) ---
        constraints = []
        constraints.append(x_var[0] == start_pos_param)
        
        for t in range(N):
            constraints.append(x_var[t+1] == x_var[t] + u_var[t] * DT)

        for t in range(N):
            constraints.append(cp.sum_squares(u_var[t]) <= (MAX_SPEED/DT)**2)
            
        for t in range(N + 1):
            constraints.append(a_param * x_var[t, 0] + b_param * x_var[t, 1] >= c_param)

        # --- Create the Problem object ---
        prob = cp.Problem(cp.Minimize(cost), constraints)
    
    # --- Solve the Problem ---
    try:
        prob.solve(solver=cp.SCS, verbose=False, warm_start=True)
        if prob.status == 'optimal' or prob.status == 'optimal_inaccurate':
            return x_var.value, (a, b, c) # Return path and the line
    except cp.error.SolverError:
        pass # Solver failed

    return None, (a, b, c)

# --- Pygame Setup ---
pygame.init()
screen = pygame.display.set_mode((800, 600))
pygame.display.set_caption("MPC with CVXPY Solver (Win/Lose)")
clock = pygame.time.Clock()

# --- Colors ---
COLOR_WHITE = (255, 255, 255)
COLOR_BLACK = (0, 0, 0)
COLOR_RED = (255, 0, 0)
COLOR_GREEN = (0, 255, 0)
COLOR_BLUE = (0, 0, 255)
COLOR_GRAY = (200, 200, 200)
COLOR_DARK_GREEN = (0, 150, 0)
COLOR_DARK_RED = (150, 0, 0)

# --- Fonts ---
font = pygame.font.Font(None, 74) # For win/lose text
small_font = pygame.font.Font(None, 30) # For stats

# --- Game Objects ---
obstacle_start_pos = (375, 275)
obstacle = pygame.Rect(obstacle_start_pos[0], obstacle_start_pos[1], 50, 50)
obstacle_pos_f = np.array(obstacle.center, dtype=float) # ADDED: Float position for obstacle
OBSTACLE_MAX_SPEED = 5.0 # ADDED: Max speed for player obstacle

target_pos_np = np.array([700.0, 500.0])
target_radius = 10
object_start_pos = np.array([100.0, 100.0])
object_pos_np = np.array([100.0, 100.0])
object_radius = 15

moving_obstacle = False
planned_path = None
debug_line = None

# --- ADDED: Game State ---
game_state = 'running' # 'running', 'win', 'lose'

def reset_game():
    """Resets the game to the starting state."""
    global object_pos_np, planned_path, game_state, obstacle, obstacle_pos_f # MODIFIED
    object_pos_np = np.array(object_start_pos)
    planned_path = None
    game_state = 'running'
    obstacle.topleft = obstacle_start_pos # Reset obstacle rect
    obstacle_pos_f = np.array(obstacle.center, dtype=float) # ADDED: Reset float position

running = True
while running:
    # --- Event handling ---
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.MOUSEBUTTONDOWN:
            if game_state == 'running':
                if obstacle.collidepoint(event.pos):
                    moving_obstacle = True
            else:
                # If game is over, click to reset
                reset_game()
        elif event.type == pygame.MOUSEBUTTONUP:
            moving_obstacle = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_r:
                reset_game()

    # --- Game Logic (Only run if game is active) ---
    if game_state == 'running':
        # --- Move obstacle (Player) ---
        if moving_obstacle:
            # --- MODIFIED: Limit obstacle speed ---
            mouse_pos_np = np.array(pygame.mouse.get_pos(), dtype=float)
            move_vec = mouse_pos_np - obstacle_pos_f
            dist_to_mouse = np.linalg.norm(move_vec)

            if dist_to_mouse > 0: # Avoid division by zero
                if dist_to_mouse <= OBSTACLE_MAX_SPEED:
                    # If mouse is close or moving slow, just snap to it
                    obstacle_pos_f = mouse_pos_np
                else:
                    # If mouse is far, move at max speed towards it
                    move_dir = move_vec / dist_to_mouse
                    obstacle_pos_f += move_dir * OBSTACLE_MAX_SPEED
            
            obstacle.center = obstacle_pos_f.astype(int) # Update the rect
            # --- END MODIFICATION ---

            planned_path = None # Invalidate path if obstacle moves

        # --- MPC Solver (The "Brain") ---
        obstacle_center_np = np.array(obstacle.center, dtype=float)
        
        path, line = solve_mpc_trajectory(object_pos_np, target_pos_np, obstacle_center_np)
        
        debug_line = line # Store line for drawing
        if path is not None:
            planned_path = path
            # Move to the *next step* in our optimal plan
            object_pos_np = planned_path[1]
        else:
            # Solver failed! Fallback: just move towards target
            dx = target_pos_np[0] - object_pos_np[0]
            dy = target_pos_np[1] - object_pos_np[1]
            dist = np.linalg.norm(target_pos_np - object_pos_np)
            if dist > 1:
                object_pos_np[0] += (dx / dist) * (MAX_SPEED * DT)
                object_pos_np[1] += (dy / dist) * (MAX_SPEED * DT)
        
        # --- ADDED: Win/Lose Condition Checks ---
        
        # 1. Win Condition (Object reaches target)
        dist_to_target = np.linalg.norm(object_pos_np - target_pos_np)
        if dist_to_target < object_radius + target_radius:
            game_state = 'win'

        # 2. Lose Condition (Object hits obstacle)
        # Find closest point on rectangle to circle's center
        closest_x = max(obstacle.left, min(object_pos_np[0], obstacle.right))
        closest_y = max(obstacle.top, min(object_pos_np[1], obstacle.bottom))
        dist_to_obs = np.linalg.norm(object_pos_np - np.array([closest_x, closest_y]))
        
        if dist_to_obs < object_radius:
            game_state = 'lose'

    # --- Draw everything ---
    screen.fill(COLOR_WHITE)
    
    # Draw the debug "keep out" line
    if debug_line:
        a, b, c = debug_line
        if abs(a) > 1e-6 and abs(b) > 1e-6:
            p1_y = 0
            p1_x = (c - b * p1_y) / a
            p2_y = 600
            p2_x = (c - b * p2_y) / a
            pygame.draw.line(screen, COLOR_GREEN, (p1_x, p1_y), (p2_x, p2_y), 2)

    # Draw the planned path
    if planned_path is not None:
        points = [tuple(p) for p in planned_path]
        pygame.draw.lines(screen, COLOR_GRAY, False, points, 2)
        
    # Draw game objects
    pygame.draw.rect(screen, COLOR_RED, obstacle)
    pygame.draw.circle(screen, COLOR_GREEN, target_pos_np.astype(int), target_radius)
    pygame.draw.circle(screen, COLOR_BLUE, object_pos_np.astype(int), object_radius)

    # --- ADDED: Draw UI (Stats or Win/Lose Text) ---
    if game_state == 'win':
        text = font.render('YOU WIN!', True, COLOR_DARK_GREEN)
        screen.blit(text, (screen.get_width() // 2 - text.get_width() // 2, 250))
        sub_text = small_font.render('Click or press R to play again', True, COLOR_BLACK)
        screen.blit(sub_text, (screen.get_width() // 2 - sub_text.get_width() // 2, 320))
    elif game_state == 'lose':
        text = font.render('YOU LOSE!', True, COLOR_DARK_RED)
        screen.blit(text, (screen.get_width() // 2 - text.get_width() // 2, 250))
        sub_text = small_font.render('Click or press R to play again', True, COLOR_BLACK)
        screen.blit(sub_text, (screen.get_width() // 2 - sub_text.get_width() // 2, 320))
    else:
        # If game is running, show stats
        fps = clock.get_fps()
        fps_text = small_font.render(f'FPS: {fps:.1f}', True, COLOR_BLACK)
        screen.blit(fps_text, (10, 10))
        solver_status = prob.status if (prob and prob.status) else "N/A"
        status_text = small_font.render(f'Solver: {solver_status}', True, COLOR_BLACK)
        screen.blit(status_text, (10, 40))

    pygame.display.flip()
    clock.tick(60)

pygame.quit()




In [3]:
import pygame
import math
import numpy as np
import cvxpy as cp

# --- CVXPY (MPC) Setup ---
N = 20              # Prediction Horizon: How many steps to plan ahead
DT = 0.1            # Time step for the plan
MAX_SPEED = 50      # Max speed (pixels per second)
MIN_DIST = 40       # Min distance from obstacle (radius + obstacle size)

# Create the CVXPY optimization variables
x_var = cp.Variable((N + 1, 2))
u_var = cp.Variable((N, 2))
prob = None

# Define CVXPY Parameters
start_pos_param = cp.Parameter(2, name="start_pos")
target_pos_param = cp.Parameter(2, name="target_pos")
a_param = cp.Parameter(name="a")
b_param = cp.Parameter(name="b")
c_param = cp.Parameter(name="c")

def solve_mpc_trajectory(start_pos, target_pos, obstacle_center):
    """
    Solves the MPC optimization problem for one frame.
    Finds the best path from start_pos to target_pos while avoiding the obstacle.
    """
    global prob
    
    # --- The "Big Trick": Linearizing the Obstacle Constraint ---
    obs_to_obj_vec = start_pos - obstacle_center
    norm = np.linalg.norm(obs_to_obj_vec)
    if norm < 1e-6:
        obs_to_obj_vec = np.array([1.0, 0.0])
        norm = 1.0
        
    a = obs_to_obj_vec[0] / norm
    b = obs_to_obj_vec[1] / norm
    c = a * (obstacle_center[0] + a * MIN_DIST) + b * (obstacle_center[1] + b * MIN_DIST)
    
    # --- Update Parameter Values ---
    start_pos_param.value = start_pos
    target_pos_param.value = target_pos
    a_param.value = a
    b_param.value = b
    c_param.value = c

    # --- Define the Problem (ONLY ONCE) ---
    if prob is None:
        cost = cp.sum_squares(x_var[N] - target_pos_param)
        cost += 0.5 * cp.sum_squares(u_var)

        constraints = []
        constraints.append(x_var[0] == start_pos_param)
        for t in range(N):
            constraints.append(x_var[t+1] == x_var[t] + u_var[t] * DT)
        for t in range(N):
            constraints.append(cp.sum_squares(u_var[t]) <= (MAX_SPEED/DT)**2)
        for t in range(N + 1):
            constraints.append(a_param * x_var[t, 0] + b_param * x_var[t, 1] >= c_param)

        prob = cp.Problem(cp.Minimize(cost), constraints)
    
    # --- Solve the Problem ---
    try:
        prob.solve(solver=cp.SCS, verbose=False, warm_start=True)
        if prob.status == 'optimal' or prob.status == 'optimal_inaccurate':
            return x_var.value, (a, b, c) # Return path and the line
    except cp.error.SolverError:
        pass # Solver failed

    return None, (a, b, c)

# --- Pygame Setup ---
pygame.init()
screen = pygame.display.set_mode((800, 600))
pygame.display.set_caption("Hybrid MPC (Brain) + Potential Field (Reflex)")
clock = pygame.time.Clock()

# --- Colors ---
COLOR_WHITE = (255, 255, 255)
COLOR_BLACK = (0, 0, 0)
COLOR_RED = (255, 0, 0)
COLOR_GREEN = (0, 255, 0)
COLOR_BLUE = (0, 0, 255)
COLOR_GRAY = (200, 200, 200)
COLOR_DARK_GREEN = (0, 150, 0)
COLOR_DARK_RED = (150, 0, 0)
COLOR_YELLOW = (255, 200, 0) # --- ADDED for Waypoint

# --- Fonts ---
font = pygame.font.Font(None, 74) # For win/lose text
small_font = pygame.font.Font(None, 30) # For stats

# --- Game Objects ---
obstacle_start_pos = (375, 275)
obstacle = pygame.Rect(obstacle_start_pos[0], obstacle_start_pos[1], 50, 50)
obstacle_pos_f = np.array(obstacle.center, dtype=float)
OBSTACLE_MAX_SPEED = 5.0

target_pos_np = np.array([700.0, 500.0])
target_radius = 10
object_start_pos = np.array([100.0, 100.0])
object_pos_np = np.array([100.0, 100.0])
object_radius = 15

# --- HYBRID CONTROL SETUP ---
# --- Fast Loop (Potential Field) Physics ---
object_vel = np.array([0.0, 0.0])
ATTRACTION_STRENGTH = 1.0    # "Pull" force towards the waypoint
REPULSION_STRENGTH = 60000.0 # "Push" force from the obstacle
AVOIDANCE_RADIUS = 100.0     # How far the "push" force is felt
ACCEL_DAMPING = 0.95         # Friction/drag

# --- Slow Loop (MPC) Planning ---
planned_path_mpc = None      # Full path from MPC (for drawing)
intermediate_waypoint = np.array(target_pos_np) # "Sub-goal" for the fast loop
WAYPOINT_INDEX = 5           # Which point on the MPC plan to aim for
MPC_REPLAN_INTERVAL = 1000   # (ms) How often to run the slow "brain"
last_mpc_time = -MPC_REPLAN_INTERVAL # Force immediate plan
debug_line = None

moving_obstacle = False
game_state = 'running' # 'running', 'win', 'lose'

def reset_game():
    """Resets the game to the starting state."""
    global object_pos_np, object_vel, planned_path_mpc, game_state
    global obstacle, obstacle_pos_f, intermediate_waypoint, last_mpc_time
    
    object_pos_np = np.array(object_start_pos)
    object_vel = np.array([0.0, 0.0])
    planned_path_mpc = None
    game_state = 'running'
    obstacle.topleft = obstacle_start_pos
    obstacle_pos_f = np.array(obstacle.center, dtype=float)
    intermediate_waypoint = np.array(target_pos_np) # Reset waypoint to final target
    last_mpc_time = -MPC_REPLAN_INTERVAL # Force immediate re-plan

running = True
while running:
    current_time = pygame.time.get_ticks()
    
    # --- Event handling ---
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False
        elif event.type == pygame.MOUSEBUTTONDOWN:
            if game_state == 'running':
                if obstacle.collidepoint(event.pos):
                    moving_obstacle = True
            else:
                reset_game()
        elif event.type == pygame.MOUSEBUTTONUP:
            moving_obstacle = False
        elif event.type == pygame.KEYDOWN:
            if event.key == pygame.K_r:
                reset_game()

    # --- Game Logic (Only run if game is active) ---
    if game_state == 'running':
        # --- Move obstacle (Player) ---
        if moving_obstacle:
            mouse_pos_np = np.array(pygame.mouse.get_pos(), dtype=float)
            move_vec = mouse_pos_np - obstacle_pos_f
            dist_to_mouse = np.linalg.norm(move_vec)

            if dist_to_mouse > 0:
                if dist_to_mouse <= OBSTACLE_MAX_SPEED:
                    obstacle_pos_f = mouse_pos_np
                else:
                    move_dir = move_vec / dist_to_mouse
                    obstacle_pos_f += move_dir * OBSTACLE_MAX_SPEED
            
            obstacle.center = obstacle_pos_f.astype(int)

        # --- "SLOW LOOP": MPC Brain (Runs ~1 time per second) ---
        if current_time - last_mpc_time > MPC_REPLAN_INTERVAL:
            last_mpc_time = current_time
            obstacle_center_np = np.array(obstacle.center, dtype=float)
            
            path, line = solve_mpc_trajectory(object_pos_np, target_pos_np, obstacle_center_np)
            
            debug_line = line # Store line for drawing
            if path is not None:
                planned_path_mpc = path # Store full path for drawing
                # Update the "sub-goal" for the fast loop
                intermediate_waypoint = planned_path_mpc[WAYPOINT_INDEX]
            # If MPC fails, 'intermediate_waypoint' just isn't updated,
            # so the fast loop keeps chasing the *last known good* waypoint.

        # --- "FAST LOOP": Potential Field Reflex (Runs 60 times per second) ---
        
        # 1. Attractive force (pulls toward intermediate waypoint)
        dx_target = intermediate_waypoint[0] - object_pos_np[0]
        dy_target = intermediate_waypoint[1] - object_pos_np[1]
        dist_target = math.hypot(dx_target, dy_target)
        
        force_attract_x = (dx_target / (dist_target + 1e-6)) * ATTRACTION_STRENGTH
        force_attract_y = (dy_target / (dist_target + 1e-6)) * ATTRACTION_STRENGTH

        # 2. Repulsive force (pushes away from obstacle)
        dx_obs = object_pos_np[0] - obstacle.centerx
        dy_obs = object_pos_np[1] - obstacle.centery
        dist_obs = math.hypot(dx_obs, dy_obs)

        force_repel_x = 0
        force_repel_y = 0
        if 0 < dist_obs < AVOIDANCE_RADIUS:
            # Force is inversely proportional to distance (stronger when closer)
            strength = REPULSION_STRENGTH / (dist_obs**2)
            force_repel_x = (dx_obs / dist_obs) * strength
            force_repel_y = (dy_obs / dist_obs) * strength
            
        # 3. Apply forces (Simple physics)
        accel_x = force_attract_x + force_repel_x
        accel_y = force_attract_y + force_repel_y
        
        object_vel[0] = (object_vel[0] + accel_x) * ACCEL_DAMPING
        object_vel[1] = (object_vel[1] + accel_y) * ACCEL_DAMPING
        
        object_pos_np += object_vel * (clock.get_time() / 1000.0) # Scale by frame time

        # --- Win/Lose Condition Checks ---
        dist_to_final_target = np.linalg.norm(object_pos_np - target_pos_np)
        if dist_to_final_target < object_radius + target_radius:
            game_state = 'win'

        closest_x = max(obstacle.left, min(object_pos_np[0], obstacle.right))
        closest_y = max(obstacle.top, min(object_pos_np[1], obstacle.bottom))
        dist_to_obs = np.linalg.norm(object_pos_np - np.array([closest_x, closest_y]))
        
        if dist_to_obs < object_radius:
            game_state = 'lose'

    # --- Draw everything ---
    screen.fill(COLOR_WHITE)
    
    # Draw the debug "keep out" line
    if debug_line and game_state == 'running':
        a, b, c = debug_line
        if abs(a) > 1e-6 and abs(b) > 1e-6:
            p1_y = 0
            p1_x = (c - b * p1_y) / a
            p2_y = 600
            p2_x = (c - b * p2_y) / a
            pygame.draw.line(screen, COLOR_GREEN, (p1_x, p1_y), (p2_x, p2_y), 2)

    # Draw the full MPC planned path
    if planned_path_mpc is not None and game_state == 'running':
        points = [tuple(p) for p in planned_path_mpc]
        pygame.draw.lines(screen, COLOR_GRAY, False, points, 2)
        
    # Draw the intermediate waypoint
    pygame.draw.circle(screen, COLOR_YELLOW, intermediate_waypoint.astype(int), 6)
        
    # Draw game objects
    pygame.draw.rect(screen, COLOR_RED, obstacle)
    pygame.draw.circle(screen, COLOR_DARK_GREEN, target_pos_np.astype(int), target_radius)
    pygame.draw.circle(screen, COLOR_BLUE, object_pos_np.astype(int), object_radius)

    # --- Draw UI (Stats or Win/Lose Text) ---
    if game_state == 'win':
        text = font.render('YOU WIN!', True, COLOR_DARK_GREEN)
        screen.blit(text, (screen.get_width() // 2 - text.get_width() // 2, 250))
        sub_text = small_font.render('Click or press R to play again', True, COLOR_BLACK)
        screen.blit(sub_text, (screen.get_width() // 2 - sub_text.get_width() // 2, 320))
    elif game_state == 'lose':
        text = font.render('YOU LOSE!', True, COLOR_DARK_RED)
        screen.blit(text, (screen.get_width() // 2 - text.get_width() // 2, 250))
        sub_text = small_font.render('Click or press R to play again', True, COLOR_BLACK)
        screen.blit(sub_text, (screen.get_Gwidth() // 2 - sub_text.get_width() // 2, 320))
    else:
        fps = clock.get_fps()
        fps_text = small_font.render(f'FPS: {fps:.1f}', True, COLOR_BLACK)
        screen.blit(fps_text, (10, 10))
        solver_status = prob.status if (prob and prob.status) else "N/A"
        status_text = small_font.render(f'Solver: {solver_status}', True, COLOR_BLACK)
        screen.blit(status_text, (10, 40))
        
        time_to_plan = MPC_REPLAN_INTERVAL - (current_time - last_mpc_time)
        plan_text = small_font.render(f'Next plan in: {time_to_plan:4.0f}ms', True, COLOR_BLACK)
        screen.blit(plan_text, (10, 70))


    pygame.display.flip()
    clock.tick(60)

pygame.quit()
